In [ ]:
%%capture
# ! pip install -U peft wandb bitsandbytes evaluate

In [ ]:
import numpy as np
import pandas as pd
import os

import torch
import torch.nn.functional as F

from datasets import Dataset
from huggingface_hub import notebook_login
import evaluate

from transformers import (AutoConfig,
                          AutoModelForSequenceClassification,
                          EarlyStoppingCallback,
                          BitsAndBytesConfig,
                          AutoTokenizer,
                          TrainingArguments,
                          DataCollatorWithPadding,
                          Trainer)

import wandb
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

In [ ]:
MAX_CHARS = 9000
# Convert columns to string and integer types to prevent errors
df_train['file_1'] = df_train['file_1'].astype(str)
df_train['file_2'] = df_train['file_2'].astype(str)
df_train['real_text_id'] = df_train['real_text_id'].astype(int)

In [ ]:
texts = []
labels = []

for _, row in df_train.iterrows():
    if row['real_text_id'] == 1:
        texts.append(row['file_1'][:MAX_CHARS])
        labels.append(1)
        texts.append(row['file_2'][:MAX_CHARS])
        labels.append(0)
    else:
        texts.append(row['file_1'][:MAX_CHARS])
        labels.append(0)
        texts.append(row['file_2'][:MAX_CHARS])
        labels.append(1)


train_data = {
    "text": texts,
    "labels": labels
}
train_dataset = Dataset.from_dict(train_data)

In [ ]:
notebook_login()

wandb.login(key="ecebfb1c0c43000ce0e6dc14a0bf61d3ab03b5e6")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sgtambrosio2019 (sgtambrosio2019-ambrosio) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint,
                                            device_map="auto",
                                            num_labels=2,
                                            torch_dtype=torch.bfloat16)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def function_tokenize(example):
    return tokenizer(
        example["text"],
        padding='max_length',
        truncation=True,
        max_length=512
    )

In [ ]:
dataset = train_dataset.train_test_split(test_size=0.1, seed=42)
dataset = dataset.map(function_tokenize, batched=True)
dataset = dataset.remove_columns(["text"])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/171 [00:00<?, ? examples/s]

Map:   0%|          | 0/19 [00:00<?, ? examples/s]

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 171
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 19
    })
})

In [ ]:
# for _  in range(10):
#     i = np.random.randint(len(dataset))
#     print(f"Text Tokenized: {dataset['input_ids'][i][:20]}")

In [ ]:
def compute_metrics(eval_preds):
    metric = evaluate.load("accuracy", "f1")
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
# Iterate through the transformer layers in the encoder
# for i, layer in enumerate(model.bert.encoder.layer):
#   print(f"Layer {i}: {layer}")

# **Training Model**

In [ ]:
output_dir = f"{model_checkpoint}-text-classification"
output_dir

'bert-base-uncased-text-classification'

In [ ]:
training_args = TrainingArguments(
    output_dir=output_dir,
    load_best_model_at_end=True,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    optim="adamw_8bit",
    num_train_epochs=100,
    bf16=True,
    seed=42,
    remove_unused_columns=False,
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
# Train model
wandb.init(project="text-classification")
trainer.train()

In [ ]:
repo_id = "bert-real-fake-classification"
trainer.model.push_to_hub(repo_id=repo_id,
                          commit_message="Training complete",
                          private=True)

model_checkpoint = f"Ambrosio1994/{repo_id}"

config = AutoConfig.from_pretrained(model_checkpoint)

config.push_to_hub(model_checkpoint, commit_message="Add config.json")

tokenizer.push_to_hub(model_checkpoint, commit_message="Add tokenizer.json")

model.safetensors:   0%|          | 0.00/219M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/Ambrosio1994/bert-real-fake-classification/commit/530e8dfbfcbb098f65af699dbde5c64dcbb9cfcf', commit_message='Add tokenizer.json', commit_description='', oid='530e8dfbfcbb098f65af699dbde5c64dcbb9cfcf', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Ambrosio1994/bert-real-fake-classification', endpoint='https://huggingface.co', repo_type='model', repo_id='Ambrosio1994/bert-real-fake-classification'), pr_revision=None, pr_num=None)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint)
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

model.safetensors:   0%|          | 0.00/219M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

In [ ]:
def inference_pipeline(text):
    inputs = tokenizer(text, truncation=True, max_length=512, return_tensors="pt")
    with torch.no_grad():
        inputs = inputs.to(device)
        logits = model(**inputs).logits
        logits = logits.detach().to('cpu')
        probs = F.softmax(logits, dim=1)
        return probs

In [ ]:
def get_real_idx(prob_fake_file1, prob_fake_file2):
    if prob_fake_file1 < prob_fake_file2:
        return 1
    else:
        return 2

In [ ]:
df_test["logits1"] = df_test.apply(lambda x: float(inference_pipeline(x["file_1"][:MAX_CHARS])[0][0]), axis=1)
df_test["logits2"] = df_test.apply(lambda x: float(inference_pipeline(x["file_2"][:MAX_CHARS])[0][0]), axis=1)

In [ ]:
df_test['real_text_id'] = df_test.apply(lambda x: get_real_idx(x['logits1'], x['logits2']), axis=1)

In [ ]:
df_sub = df_test[['id', 'real_text_id']]
df_sub.to_csv("submission_04_07_sub3.csv", index=False)
df_sub

,id,real_text_id
0,0,1
1,1,2
2,2,2
3,3,1
4,4,1
...,...,...
1063,1063,2
1064,1064,1
1065,1065,1
1066,1066,2
